In [13]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression

In [ ]:
# run from the competition folder; train.csv is downloaded from the Kaggle competition page
X_train = pd.read_csv('train.csv')
X_train = X_train.dropna(subset=["target"]).reset_index(drop=True)

In [15]:
def time_series_folds(date_id, n_splits=5, first_train_days=100, embargo=10):
    date_id = np.asarray(date_id)
    unique_dates = np.sort(np.unique(date_id))

    # first validation block starts once first_train_days of history exist
    first_val_start = unique_dates[0] + first_train_days
    val_pool = unique_dates[unique_dates >= first_val_start]

    # contiguous, non-overlapping validation blocks tiling the remaining timeline
    # (np.array_split gives the leftover days to the final block)
    for block in np.array_split(val_pool, n_splits):
        val_start = block[0]
        # embargo gap sits immediately before the validation block
        train_dates = unique_dates[unique_dates < val_start - embargo]

        train_idx = np.nonzero(np.isin(date_id, train_dates))[0]
        valid_idx = np.nonzero(np.isin(date_id, block))[0]
        yield train_idx, valid_idx

In [16]:
X_train = X_train.sort_values(['stock_id', 'date_id', 'seconds_in_bucket']).reset_index(drop=True)

In [17]:
y_train = X_train['target'].copy()
date_id = X_train['date_id'].copy()

In [18]:
X_train['signed_imbalance'] = X_train['imbalance_size'] * X_train['imbalance_buy_sell_flag']

g = X_train.groupby(["stock_id", "date_id"], sort=False)

current = X_train['signed_imbalance']

X_train['imbalance_delta_1b'] = current - g['signed_imbalance'].shift(1)
X_train['imbalance_delta_2b'] = current - g['signed_imbalance'].shift(2)

X_train['normalise_imb_del_1'] = X_train['imbalance_delta_1b'] / (X_train['matched_size'].replace(0, np.nan))
X_train['normalise_imb_del_2'] = X_train['imbalance_delta_2b'] / (X_train['matched_size'].replace(0, np.nan))

X_train['stock_id'] = X_train['stock_id'].astype('category')



In [19]:
X_train['log_wap'] = np.log(X_train['wap'])

X_train['wap_ret_1b'] = g['log_wap'].diff(1)          # log return over the last 10s
X_train['wap_ret_2b'] = g['log_wap'].diff(2)          # over the last 20s
X_train['wap_ret_3b'] = g['log_wap'].diff(3)

X_train['wap_rv_6b'] = (
    g['wap_ret_1b'].rolling(6, min_periods=3).std().reset_index(level=[0, 1], drop=True)
)

In [20]:
X_train.drop(columns=['date_id', 'row_id', 'time_id', 'target'], inplace=True, errors='ignore')

In [21]:
# reuse the same splitter so the fold date-ranges match your training run exactly
for i, (train_idx, valid_idx) in enumerate(time_series_folds(date_id.values), 1):
    va = y_train.iloc[valid_idx]                      # this fold's validation targets

    target_std = va.std()                             # spread of the target in this period
    mae_zero = va.abs().mean()                        # MAE of a constant "predict 0" model
    mae_median = (va - va.median()).abs().mean()      # MAE of "predict the training median"

    print(f"fold {i}: n={len(va):>8}  std={target_std:.3f}  "
          f"MAE(0)={mae_zero:.3f}  MAE(median)={mae_median:.3f}")

# Previous results:
# fold 1: n=  834240  std=10.329  MAE(0)=7.205  MAE(median)=7.205
# fold 2: n=  830830  std=10.081  MAE(0)=6.921  MAE(median)=6.920
# fold 3: n=  833634  std=9.023  MAE(0)=6.244  MAE(median)=6.244
# fold 4: n=  835945  std=9.571  MAE(0)=6.432  MAE(median)=6.431
# fold 5: n=  835999  std=8.807  MAE(0)=5.928  MAE(median)=5.928


fold 1: n=  834240  std=10.329  MAE(0)=7.205  MAE(median)=7.205
fold 2: n=  830830  std=10.081  MAE(0)=6.921  MAE(median)=6.920
fold 3: n=  833634  std=9.023  MAE(0)=6.244  MAE(median)=6.244
fold 4: n=  835945  std=9.571  MAE(0)=6.432  MAE(median)=6.431
fold 5: n=  835999  std=8.807  MAE(0)=5.928  MAE(median)=5.928


In [22]:
X_train.drop(columns=['wap_rv_6b', 'imbalance_delta_1b', 'imbalance_delta_2b', 'log_wap'], inplace=True, errors='ignore')

In [23]:
scores = []
gains = []
best_iters = []                                          # early-stopped tree count per fold
for i, (train_idx, valid_idx) in enumerate(time_series_folds(date_id.values), 1):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    model = lgb.LGBMRegressor(objective='regression_l1', n_estimators=500, learning_rate=0.05, max_depth=8, random_state=42, importance_type='gain')
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='mae', callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    model_mae = float(model.best_score_['valid_0']['l1'])   # MAE on this fold's validation block
    mae_zero = float(y_va.abs().mean())                      # MAE of predicting 0 everywhere (naive floor)
    gain = mae_zero - model_mae                             # signal extracted over predict-0; compare THIS across feature experiments, not raw MAE
    gains.append(gain)
    best_iters.append(model.best_iteration_)
    scores.append({'fold': i, 'model_mae': model_mae, 'mae_zero': mae_zero, 'gain': gain, 'best_iter': model.best_iteration_})
    print(f"fold {i}: model_mae={model_mae:.4f}  mae_zero={mae_zero:.4f}  gain={gain:.4f}  best_iter={model.best_iteration_}")
    
scores = pd.DataFrame(scores)
# mean gain = overall signal; worst gain over folds 2-5 (fold 1 trains on only 100 days) = robustness check
print(f"\nmean gain={scores['gain'].mean():.4f}  worst gain (folds 2-5)={scores['gain'].iloc[1:].min():.4f}")
print(f'Gains from each fold: {gains}')
print(f'Best iters per fold: {best_iters}')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006294 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 958894, number of used features: 19
[LightGBM] [Info] Start training from score -0.060201
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[177]	valid_0's l1: 7.07229
fold 1: model_mae=7.0723  mae_zero=7.2052  gain=0.1329  best_iter=177


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011083 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 1793134, number of used features: 19
[LightGBM] [Info] Start training from score -0.069737
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[190]	valid_0's l1: 6.8188
fold 2: model_mae=6.8188  mae_zero=6.9208  gain=0.1020  best_iter=190


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015908 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4337
[LightGBM] [Info] Number of data points in the train set: 2622864, number of used features: 19
[LightGBM] [Info] Start training from score -0.079870
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[162]	valid_0's l1: 6.17635
fold 3: model_mae=6.1764  mae_zero=6.2435  gain=0.0672  best_iter=162


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022240 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4338
[LightGBM] [Info] Number of data points in the train set: 3456004, number of used features: 19
[LightGBM] [Info] Start training from score -0.060201
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[405]	valid_0's l1: 6.33977
fold 4: model_mae=6.3398  mae_zero=6.4317  gain=0.0919  best_iter=405


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044653 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4338
[LightGBM] [Info] Number of data points in the train set: 4291893, number of used features: 19
[LightGBM] [Info] Start training from score -0.069737
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[465]	valid_0's l1: 5.84695
fold 5: model_mae=5.8469  mae_zero=5.9280  gain=0.0810  best_iter=465

mean gain=0.0950  worst gain (folds 2-5)=0.0672
Gains from each fold: [0.13292210543913807, 0.1020125463512569, 0.06719685838102496, 0.09190965566425291, 0.081013351144267]
Best iters per fold: [177, 190, 162, 405, 465]


In [24]:
fi = pd.Series(model.feature_importances_, index=X_tr.columns).sort_values(ascending=False)
print(fi)

stock_id                   863757.516880
ask_size                   306310.809949
bid_size                   283001.790429
seconds_in_bucket          132954.015476
reference_price             60593.084602
wap                         57153.264753
signed_imbalance            53180.823444
wap_ret_1b                  48512.037793
near_price                  41494.216090
normalise_imb_del_1         40071.008924
wap_ret_3b                  34684.240688
normalise_imb_del_2         31002.562900
wap_ret_2b                  26921.200624
bid_price                   15349.507288
ask_price                   15055.971593
imbalance_size              13020.424435
far_price                   11423.439490
matched_size                 9685.510182
imbalance_buy_sell_flag      4167.364931
dtype: float64


In [ ]:
# --- final model: retrain on ALL data with the tree count fixed from CV ---
# No validation set here, so we can't early-stop. Fold 5 trains on ~90% of the rows
# (closest to this full-data fit); take its best_iter plus a ~15% uplift for the extra data.
# A median across folds would underfit — the small early folds converge in ~170 trees.
n_est = int(best_iters[-1] * 1.15)

final_model = lgb.LGBMRegressor(objective='regression_l1', n_estimators=n_est, learning_rate=0.05,
                                max_depth=8, random_state=42)
final_model.fit(X_train, y_train)
print(f'trained final_model on {len(X_train)} rows with n_estimators={n_est}')
